In [ ]:
# !pip install python-dotenv
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# read csv
logistics_data = pd.read_csv("data/anonymized_logistics_picking_data.csv", sep=";")

# 1. Data Cleaning
## 1.1 Check structure abd datatypes

In [ ]:
logistics_data.info()

## 1.2 Data Type Conversion

The raw dataset contains several columns stored in incorrect formats (e.g., numeric and date fields saved as text). 

In this step, numeric columns such as `volume` and `wm_vsola` are converted to proper numeric types, and date/time fields (`calday`, `start_time`, `finish_time`) are converted to datetime format. 

This ensures accurate calculations, especially for duration analysis and future aggregations.

In [ ]:
df = logistics_data.copy()

# 1️. Fix Volume (comma decimal → dot)
df["volume"] = df["volume"].str.replace(",", ".", regex=False)
df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

# 2️. Convert wm_vsola to numeric
df["wm_vsola"] = pd.to_numeric(df["wm_vsola"], errors="coerce")

# 3️. Convert date columns
df["calday"] = pd.to_datetime(df["calday"], errors="coerce")
df["calmonth"] = pd.to_datetime(df["calmonth"], errors="coerce")
df["calyear"] = pd.to_datetime(df["calyear"], errors="coerce")

# 4️. Convert timestamps
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["finish_time"] = pd.to_datetime(df["finish_time"], errors="coerce")

print("Data types after conversion:\n")
print(df.dtypes)

print("\nMissing values check (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

## 1.3 Create Picking Duration and Remove Invalid Records


In [ ]:
# 1. create duration in seconds
df["duration_sec"] = (df["finish_time"] - df["start_time"]).dt.total_seconds()

print("shape before cleaning", df.shape)

In [ ]:
# 2. remove rows with missing start_time
df = df[df["start_time"].notna()]

# 3. remove zero or negative durations
df = df[df["duration_sec"] > 0]

print("shape after cleaning", df.shape)

# Check how many rows were removed
print("Rows removed:", logistics_data.shape[0] - df.shape[0])

In this step, we calculate the picking duration in seconds using the difference between `finish_time` and `start_time`.

Records with missing timestamps or non-positive durations are removed, as they likely represent system errors or invalid scanning activity. 

This ensures that the dataset reflects only valid picking tasks for accurate analysis.

## 1.4 Fix Invalid Volume Values


In [ ]:
#count zero or negative volume values
invalid_volumes = (df["volume"] <= 0).sum()

print("Number of invalid volumes :", invalid_volumes)

In [ ]:
# 1. Set invalid volumes to NaN
df.loc[df["volume"] <= 0, "volume"] = np.nan

# Check missing volume before imputation
print("Missing volume before:", df["volume"].isna().sum())


In [ ]:
# 2. Impute using median volume per product
df["volume"] = df.groupby("product_id") ["volume"]\
                    .transform(lambda x:x.fillna(x.median()))

# Check missing volume after
print("Missing volume after:", df["volume"].isna().sum())

Volume is a key operational variable influencing picking duration.

Invalid values (zero, negative, or missing) are replaced using the median volume within each product group. This approach preserves product-specific size characteristics, since different products (e.g., apples vs. bananas) naturally vary in volume. Using product-level medians ensures more realistic and operationally meaningful estimates compared to applying a global average.


## 1.5 Create Analytical Features

In [ ]:
# 1️. Extract Date from start_time
df["date"] = df["start_time"].dt.date

# 2️. Weekday Name
df["weekday_name"] = df["start_time"].dt.day_name()

# 3️. Weekend Flag
df["is_weekend"] = df["weekday_name"].isin(["Saturday", "Sunday"])

# 4️. Extract Location Components
df["location_row"] = df["location"].str[3:5].astype(int)
df["location_id"] = df["location"].str[6:9].astype(int)

print("New columns added successfully.")

In [ ]:
print(df[["date", "weekday_name", "is_weekend", "location_row", "location_id"]].head())

Additional analytical fields are created to support business reporting and warehouse modeling.

Date-related fields enable time-based analysis (weekday trends and weekend comparison), while location components allow performance evaluation by warehouse zone. These features will later form the basis of dimension tables in the data warehouse.

In [ ]:
print(df.head())

## 1.6 Remove Unnecessary Columns
To simplify the dataset for analytical and BI purposes, we remove system-generated and non-informative columns. 

Columns such as `area`, `warehouse`, `wm_srsrc`, `wm_qdocno`, and `wm_tapos` do not contribute to operational performance analysis and are therefore excluded. 

Key operational variables like `product_id`, `collli_id`, `wm_vsola`, and location identifiers are retained to allow deeper analysis if needed.

In [ ]:
# Drop columns that are not useful for BI/analytics
columns_to_drop = [
    "area",
    "warehouse",
    "calweek",
    "calmonth",
    "calyear",
    "wm_tapos",
    "wm_srsrc",   # all missing
    "wm_qdocno"   # almost all missing
]

df = df.drop(columns=columns_to_drop)

print("Remaining columns:")
print(df.columns.tolist())


## 1.7 Export Clean Analytical Dataset

After cleaning and removing unnecessary columns, the dataset is ready for analytical use.

The cleaned dataset will serve as the staging layer for SQL data warehouse modeling and Power BI reporting. This ensures consistent and reproducible data transformations.

In [ ]:
# Final dataset shape
print("Final dataset shape:", df.shape)

# Save clean dataset inside data folder
df.to_csv("data/clean_picking_data.csv", index=False)

print("Clean dataset saved in data/ folder")

## 1.8 Connect to database

In [ ]:
#install packages
!pip install sqlalchemy psycopg2-binary pandas

In [ ]:
# connect jupyter to PostgreSQL
from sqlalchemy import create_engine

# Databse connection details
username = os.getenv("PG_USER")
password = os.getenv("PG_PASSWORD")
host = os.getenv("PG_HOST")
port = os.getenv("PG_PORT")
database = os.getenv("PG_DATABASE")

# Create PostgreSQL engine
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# test connection
with engine.connect() as conn:
    print("Connected to PostgreSQL!")

In [ ]:
# load dataframe to Postgres
df = pd.read_csv("data/clean_picking_data.csv")

# Load into public schema/table (auto-creates table)
df.to_sql(
    "clean_picking_data",
    engine,
    schema="public",
    if_exists="replace",   # use "append" if we want to add rows later
    index=False
)

print("Loaded into PostgreSQL: database.clean_picking_data")

The cleaned dataset is loaded into PostgreSQL to establish a structured data warehouse environment. This enables SQL-based modeling and prepares the data for dimensional design and BI reporting.